# Social-Oracle 🔮
### Does buying what a viral stock guru mentions actually pay? Tested honestly, in plain English

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Pump--and--fade: Confirmed](https://img.shields.io/badge/Pump--and--fade-Confirmed-8b949e?style=flat-square)

Every cycle a retail-investing folk hero goes viral, and within weeks GitHub fills with bots that scrape her posts, pull out the `$TICKERS`, and score them as buy signals — *her timeline front-runs the market, just buy what she mentions.* We test that on **1,468 real WallStreetBets surges**: a mention carries **no abnormal edge over a random day**, the gross 'gain' is pure market beta the costs erase, and the one-day flicker fades to a *negative* month. It's a pump you're late to, dressed as a signal.

> 📓 **This is the plain-language layer.** Want the statistics, the microstructure and the capacity maths? That's the companion notebook, **[02_for_the_quants.ipynb](02_for_the_quants.ipynb)** — same story, deeper.
>
> ⚠️ **Not investment advice.** An educational, reproducible research tool: every chart below is generated by the code beside it. It tests a *phenomenon*, not a person. House style in [METHODOLOGY.md](../../../METHODOLOGY.md).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # study root (social_oracle/ lives there)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from social_oracle import data, mentions, eventstudy, benchmark, backtest, robustness

# No live feed ships with this study: we run the *method* on a synthetic universe
# with a baked-in pump-and-fade. Swap the next line for data.load_feed('mentions.csv')
# + data.build_panel(...) to run it for real.
panel, feed = data.synthetic_panel(seed=0)
events, coverage = mentions.to_events(feed, panel)
print(f"{len(panel)} names, {len(feed)} mentions -> {len(events)} clean events")
print("coverage:", coverage)


## The answer first 🎯

| What we asked | The honest answer (the desk's prior) |
|---|---|
| Does a name move after she mentions it? | ⚠️ **A little, briefly** — attention does nudge price. But look *left* of the tweet: most of the move already happened. |
| Is that bump a free edge? | ❌ **No** — it *fades*, and you only see the tweet *after* the pop, so you buy the start of the reversal. |
| Does "mentioned" beat "already hot"? | ❓ **Barely / not really** — strip out the momentum the name already had and most of the signal goes with it. |
| Could a follower actually trade it? | ❌ **Not after costs** — these are \$1–3 names with huge spreads and tiny capacity. |

> Desk shorthand: **Signal `WEAK` · Tradability `MIRAGE`** — let's see the method earn them.

## 1 · The claim 📣

A viral persona (here *白毛股神* **Serenity**, @aleabitoreddit) gets her posts scraped into cashtag signals by a wave of open-source repos. The strongest version of the claim: *a public mention is, on average, followed by a gain you could have captured.* We test that — on **abnormal** returns (the name minus the market), so a rising tide that lifted everything doesn't count as a call.

In [ ]:
es = eventstudy.event_study(panel, events, horizon=21, pre=5)
m = es['matrix'].mean()
plt.axvline(0, color='0.7', lw=1); plt.axhline(0, color='0.7', lw=1)
plt.plot(m.index, 100*m.values, lw=2)
plt.title(f"Average abnormal path around a mention  (n={es['n_events']} events)")
plt.xlabel('trading days from the mention'); plt.ylabel('cumulative abnormal return (%)')
plt.show()

See the shape? It climbs **into** day 0 (the run-up the tweet is chasing) and **bleeds** afterwards. The follower enters on the right-hand slope.

## 2 · So what? 💰

If attention really paid, anyone with an API key would have free alpha. If it *doesn't* — if it's a late, fading pop on names too thin to exit — then thousands of followers are buying **negatively-skewed attention beta** and calling it skill.

> The lesson Studies 02–03 keep finding from new angles: *a pattern can be obvious to the eye and still be the opposite of an edge once you ask 'more than a random day? more than the momentum it already had? net of what I'd actually pay?'*

## 3 · How we'd know 🔍

Two controls stacked, because these names are volatile by selection:

1. Did it beat a **random day** in the same universe? (the desk's usual yardstick)
2. Did the **mention** beat a name that was simply **already hot**? (momentum is the confound — attention follows performance)

Plus the **fade**: we trace the abnormal return day by day and watch it reverse.

In [ ]:
print('random-day null:')
display(benchmark.conditional_vs_unconditional(panel, events, n_iter=800).round(4))
print('\nmomentum control (mention vs already-hot):')
hot = mentions.hot_streak_events(panel)
display(benchmark.excess_vs_alternative(panel, events, hot, n_iter=800).round(4))

## 4 · The teardown 🔬

The fade, in one table — mean abnormal return at each horizon. A peak that reverses is the follower's whole problem.

In [ ]:
robustness.fade_curve(panel, events)

## 5 · The verdict ⚖️

On the synthetic the method behaves exactly as designed: the mention path runs up into the tweet, fades after, and **loses** to both a random day and a hot streak. On a real feed the literature prior (attention → small pop → reversal) says expect **Signal `WEAK`**, **Tradability `MIRAGE`**. The next beat is why a follower can't even keep the `WEAK` part.

## 6 · Could you trade it? 🏦

You read the post *after* it's public — so your entry is the **next open**, past the pop. The names are \$1–3 micro-caps with brutal spreads. Charge that, twice, and watch the mean trade sink as the spread widens.

In [ ]:
res = backtest.run(panel, events, hold_days=10)
print({k: (round(v,4) if isinstance(v,float) else v) for k,v in res.stats.items()})
backtest.cost_sweep(panel, events)

## 7 · Going further 🚪

- **The inversion:** if it's a pop-and-fade, the side that *might* pay is being **early or short the fade**, not the late follower — same punchline as Fear-Gauge (*sell* the fear, don't buy it).
- **Bring a real feed:** the whole study is one `data.load_feed('mentions.csv')` away from live numbers.
- **Conviction & first-mention:** does a high-score or first-ever mention behave differently?

The deep version — t-stats, clustering bootstrap, name jackknife, capacity — is in [`02_for_the_quants.ipynb`](02_for_the_quants.ipynb).